# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset schema is provided via a Croissant JSON-LD URL.

In [ ]:
# Ensure the latest mlcroissant library is installed
!pip install --quiet --upgrade mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets, and for each, show fields and their `@id`s.

In [ ]:
# List all record sets by ID and name
record_sets = dataset.record_sets
if not record_sets:
    print('No record sets were found in the metadata.')
else:
    print(f"Found {len(record_sets)} RecordSet(s):\n")
    for rs in record_sets:
        print(f"- RecordSet name: {rs.name} | @id: {rs.id}")
        if hasattr(rs, 'fields') and rs.fields:
            print('  Fields:')
            for field in rs.fields:
                print(f"    - {field.name} (@id: {field.id})")
        else:
            print('  No fields detected in this RecordSet.')
        print()

## 3. Data Extraction
Load records from each record set into a Pandas DataFrame for analysis. Use the record set and field `@id`s from above.

In [ ]:
# Build a dict of record_set_id -> DataFrame
dfs = {}
record_set_ids = [rs.id for rs in record_sets]

for record_set_id in record_set_ids:
    print(f'Loading records for RecordSet: {record_set_id}')
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dfs[record_set_id] = df
            print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        else:
            print('  No records found.')
    except Exception as ex:
        print(f'  Error loading record set {record_set_id}: {ex}')
print('\n')
# Preview one (first) DataFrame
if dfs:
    record_set_id = list(dfs.keys())[0]
    print(f'Preview of data in {record_set_id}:')
    display(dfs[record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps such as filtering records, normalizing numeric fields, and grouping/categorizing data.
We will select a record set with numeric fields, filter records, normalize a numeric field, and group by a relevant field. All references are to the field's `@id`.

In [ ]:
# Choose a record set for EDA. Adjust these IDs based on the overview above.
if dfs:
    record_set_id = list(dfs.keys())[0]
    df = dfs[record_set_id]
    numeric_columns = df.select_dtypes(include='number').columns.tolist()
    print(f'Numeric columns in {record_set_id}: {numeric_columns}')
    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        print(f"Using '{numeric_field_id}' for numeric EDA.")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() else 10

        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        import numpy as np
        filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

        # Try grouping by a suitable (non-numeric) field
        non_numeric_columns = [c for c in df.columns if c != numeric_field_id and df[c].dtype == 'object']
        group_field = non_numeric_columns[0] if non_numeric_columns else None
        if group_field:
            grouped = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of '{numeric_field_id}' by '{group_field}':")
            display(grouped.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("There are no numeric columns available for analysis in this record set.")
else:
    print("No record sets with DataFrames were loaded.")

## 5. Visualization
Visualize the distribution of a numeric field, and relationship to a group/categorical field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dfs and numeric_columns:
    # Plot numeric distribution (histogram)
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group field available, plot boxplot by group
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization skipped: No suitable numeric columns found.")

## 6. Conclusion
In this notebook, we've demonstrated the typical workflow for exploring a FAIR dataset with a Croissant schema via the `mlcroissant` Python API. We loaded metadata, inspected available record sets and fields (by their `@id`), extracted data, and performed initial exploratory data analysis.  
Use this template to further analyze and visualize other FAIR datasets that expose Croissant-formatted metadata.